In [1]:
# Run this if facing issue with transformers
#pip install torch torchvision torchaudio transformers

import pandas as pd
import numpy as np
import torch
from transformers import BertForSequenceClassification, BertTokenizer, AutoTokenizer, AutoModelForSequenceClassification, pipeline, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from datasets import Dataset
import tiktoken

In [3]:
news_data_final_train_labelled_df = pd.read_csv('Data/news_data_final_train_labelled.csv')
news_data_final_test_labelled_df = pd.read_csv('Data/news_data_final_test_labelled.csv')

#### Preprocessing headlines

In [4]:
# Remove ticker names within headlines
news_data_final_train_labelled_df['title'] = news_data_final_train_labelled_df['title'].str.replace(r"\('\w+':'\w+'\)", "", regex=True)
news_data_final_test_labelled_df['title'] = news_data_final_test_labelled_df['title'].str.replace(r"\('\w+':'\w+'\)", "", regex=True)

# REMOVE LATER - for testing labelling only
news_data_final_train_labelled_df = news_data_final_train_labelled_df.sample(100)
news_data_final_test_labelled_df = news_data_final_test_labelled_df.sample(100)

# Prepare df for labelling results
news_data_test_results = news_data_final_test_labelled_df.drop(columns=['source','topic'])
news_data_test_results = news_data_test_results[['date', 'Ticker', 'title', 'sentiment_label']]

#### FinBERT (Fine Tuned for Sentiment Analysis)

In [5]:
model = BertForSequenceClassification.from_pretrained('yiyanghkust/finbert-tone', num_labels=2, ignore_mismatched_sizes=True)
tokenizer = BertTokenizer.from_pretrained('yiyanghkust/finbert-tone')
model.config.problem_type = "single_label_classification"

# Tokenize data
def tokenize_data(example):
    return tokenizer(example['title'], padding='max_length',  max_length=128, truncation=True)

news_data_train_tokenized_df = Dataset.from_pandas(news_data_final_train_labelled_df)
news_data_train_tokenized_df = news_data_train_tokenized_df.map(lambda x: {'labels': x['sentiment_label']})
news_data_train_tokenized_df = news_data_train_tokenized_df.map(tokenize_data, batched=True)
news_data_train_tokenized_df.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

news_data_test_tokenized_df = Dataset.from_pandas(news_data_final_test_labelled_df)
news_data_test_tokenized_df = news_data_test_tokenized_df.map(lambda x: {'labels': x['sentiment_label']})
news_data_test_tokenized_df = news_data_test_tokenized_df.map(tokenize_data, batched=True)
news_data_test_tokenized_df.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

# Define metrics
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    return {
        'accuracy': accuracy_score(labels, predictions),
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

# Use default training hyperparameters
training_args = TrainingArguments(output_dir='test_trainer', eval_strategy='no')

# To consider using eval dataset
#training_args = TrainingArguments(output_dir='test_trainer', eval_strategy='epoch')

# OR use specific training hyperparameters
# training_args = TrainingArguments(
#     output_dir='test_trainer',
#     evaluation_strategy='epoch',
#     save_strategy='epoch',
#     learning_rate=2e-5,
#     per_device_train_batch_size=32,
#     per_device_eval_batch_size=32,
#     num_train_epochs=3,
#     weight_decay=0.01,
#     load_best_model_at_end=True,
#     metric_for_best_model='accuracy',
# )

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=news_data_train_tokenized_df,
    compute_metrics=compute_metrics
)

trainer.train()
#trainer.evaluate()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at yiyanghkust/finbert-tone and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([3, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
- classifier.bias: found shape torch.Size([3]) in the checkpoint and torch.Size([2]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

  0%|          | 0/39 [00:00<?, ?it/s]

{'train_runtime': 39.8949, 'train_samples_per_second': 7.52, 'train_steps_per_second': 0.978, 'train_loss': 0.019681557630881284, 'epoch': 3.0}


TrainOutput(global_step=39, training_loss=0.019681557630881284, metrics={'train_runtime': 39.8949, 'train_samples_per_second': 7.52, 'train_steps_per_second': 0.978, 'total_flos': 19733329152000.0, 'train_loss': 0.019681557630881284, 'epoch': 3.0})

In [6]:
finbert_predictions = trainer.predict(test_dataset=news_data_test_tokenized_df)
finbert_logits = finbert_predictions.predictions
finbert_predicted_labels = np.argmax(finbert_logits, axis=1)

news_data_test_results['finbert_sentiment_label'] = finbert_predicted_labels

  0%|          | 0/13 [00:00<?, ?it/s]